In [ ]:
import pandas as pd
import os 
import re
import math

def coef(n):
    c_n = (-1)**n/(2**n*math.factorial(n))*math.sqrt(math.factorial(2*n+1)/(4*math.pi))
    return 1/((2*n+1)*c_n**2)

# 读取文件并跳过前四行
file_path = "/home/yli/MR_CDFT/MRCDFT_dev/MRCDFT_sens/examples/150Nd/150Nd_forcefactors.dat"
kernel_dir = "/home/yli/MR_CDFT/MRCDFT_dev/MRCDFT_sens/examples/150Nd/output"
data = pd.read_csv(file_path, skiprows=4, delim_whitespace=True)
# 添加 index 列，从 1 开始
data.insert(0, 'index', range(1, len(data) + 1))

data = data[data['index']<291]

# 循环生成文件名
for i in data.index:
    idx = data.loc[i, 'index']
    beta2 = data.loc[i, 'beta2']
    # 转换成 +027、-015、+000 格式
    beta2_str = f"{int(beta2 * 100):+04d}"

    CDFT_filename = f"CDFT_150Nd_eMax12_{idx}{beta2_str}+000.out"
    Kernel_filename = (
        f"Proj_150Nd_kern.1D_eMax12.07.12_"
        f"{idx}{beta2_str}+000_"
        f"{idx}{beta2_str}+000.elem"
    )
    Ecce_filename = (
        f"Proj_150Nd_Eccen.1D_eMax12.07.12_"
        f"{idx}{beta2_str}+000_"
        f"{idx}{beta2_str}+000.elem"
    )

    CDFT_path = os.path.join(kernel_dir,CDFT_filename)
    Kernel_path = os.path.join(kernel_dir,Kernel_filename)
    Ecce_path = os.path.join(kernel_dir,Ecce_filename)

    if not os.path.exists(CDFT_path):
        print(f'{CDFT_path} does not exist.')
        continue
    if not os.path.exists(Kernel_path):
        print(f'{Kernel_path} does not exist.')
        continue
    if not os.path.exists(Ecce_path):
        print(f'{Ecce_path} does not exist.')
        continue
    

    with open(CDFT_path, 'r') as f:
        for line in f:
            if 'Iteration converged after' in line:
                words = line.split()
                steps = words[words.index('after') + 1]
                break
        for line in f:
            if line.strip().startswith('beta  ...............'):
                # 按空白字符分割
                parts = line.split()
                # 最后一个元素就是最后一列
                beta2 = parts[-1]
                break
        else:
            print("未找到以 beta 开头的行")
    with open(Kernel_path, 'r') as f:
        J = 0
        lines = f.readlines()
        line = lines[J*7+1]
        Norm = float(line[:15].strip())
        # Energy = float(line[30:45].strip())

        line = lines[J*7+4]
        r2_proton = float(line[:15].strip())
        line = lines[J*7+6]
        r2_neutron = float(line[:15].strip())
        r2 = (r2_proton + r2_neutron)/Norm

        line = lines[J*7+3]
        BE2_unnorm = float(line[:15].strip())

        J = 2
        line = lines[J*7+1]
        Norm_2 = float(line[:15].strip())

        BE2 = BE2_unnorm**2/(Norm*Norm_2)/10000



    with open(Ecce_path, 'r') as f:
        lines = f.readlines()
        line = lines[1]
        oneBody = float(line[:15].strip())
        twoBody = float(line[15:30].strip())
        line = lines[2]
        twoBody_direct = float(line[:15].strip())
        twoBody_exchange = float(line[15:30].strip())
        twoBody_kappa = float(line[30:45].strip())
        # line = lines[1]
        # values = re.findall(r'[+-]?\d+\.\d+E[+-]\d+', line)
    data.loc[i, 'It'] = steps
    data.loc[i, 'beta2'] = beta2
    data.loc[i, 'E2_1B'] = oneBody*coef(2)
    data.loc[i, 'E2_2B'] = twoBody*coef(2)
    data.loc[i, 'r2']    = r2
    data.loc[i, 'BE2(0->2)']   = BE2

In [7]:
data.to_csv(os.path.join("/home/yli/MR_CDFT/MRCDFT_dev/MRCDFT_sens/examples/150Nd/",'Nd150_PCPK1_Ecc.csv'), index=False)